# Legal IR - Improved Pipeline
## Các cải tiến so với bản gốc:
1. **Chunking cải tiến**: Thêm preamble chunk, tăng chunk size (350 words), tăng overlap (50 words)
2. **Rerank tất cả chunks**: Không dedupe trước rerank — rerank ALL chunks rồi mới dedupe
3. **Multi-chunk aggregation**: Doc có nhiều chunk match → bonus score
4. **Query expansion (PRF)**: Dùng pseudo-relevance feedback để mở rộng BM25 query
5. **HYBRID_TOP_N = 80**: Tăng từ 30, giảm retrieval miss
6. **MAX_SEQ_LENGTH = 2048**: Tận dụng tối đa capacity của AITeamVN/Vietnamese_Embedding
7. **Lambda grid search mịn hơn**: Tune chính xác hơn

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
!pip install -q rank_bm25 pyvi sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 85.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.2 MB/s eta 0:00:00


In [3]:
import re
import gc
import json
import glob
import pickle
import random
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from pyvi import ViTokenizer
from sentence_transformers import SentenceTransformer, CrossEncoder, util

print("CUDA available:", torch.cuda.is_available())
print("Số GPU:", torch.cuda.device_count())

CUDA available: True
Số GPU: 2


## Configuration

In [4]:
KAGGLE_INPUT_DATA_DIR = "/kaggle/input/datasets/thinhng04/legalir-dataset"
CONTEXT_DIR = os.path.join(KAGGLE_INPUT_DATA_DIR, "selected-contexts/selected-contexts")
TRAIN_FILE = os.path.join(KAGGLE_INPUT_DATA_DIR, "train.json")
PUBLIC_TEST_FILE = os.path.join(KAGGLE_INPUT_DATA_DIR, "public-official.json")

#CACHED_INDEX_INPUT_DIR = None  # Set None để build index mới với chunking cải tiến
CACHED_INDEX_INPUT_DIR = "/kaggle/input/datasets/thinhng04/result-cache"

WORKING_DIR = "/kaggle/working"
INDEX_DIR = os.path.join(WORKING_DIR, "index_cache")
OUTPUT_DIR = os.path.join(WORKING_DIR, "outputs")
SUBMISSION_FILE = os.path.join(OUTPUT_DIR, "submission.json")
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "submission_checkpoint.json")

os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


def resolve_index_read_dir():
    if CACHED_INDEX_INPUT_DIR is not None:
        cached_file = os.path.join(CACHED_INDEX_INPUT_DIR, "dense_embeddings.pt")
        if os.path.exists(cached_file):
            return CACHED_INDEX_INPUT_DIR
    return INDEX_DIR


DENSE_MODEL_NAME = "AITeamVN/Vietnamese_Embedding"
RERANKER_MODEL_NAME = "AITeamVN/Vietnamese_Reranker"
MAX_SEQ_LENGTH = 2048         # CẢI TIẾN: 1024 → 2048 (model hỗ trợ tới 2048, tận dụng tối đa)
RERANKER_MAX_LENGTH = 1024

GPU_DEVICES = []
for i in range(torch.cuda.device_count()):
    GPU_DEVICES.append(f"cuda:{i}")
if len(GPU_DEVICES) == 0:
    GPU_DEVICES = ["cpu"]
PRIMARY_DEVICE = GPU_DEVICES[0]

# ======================== CHUNKING (CẢI TIẾN) ========================
MAX_WORDS_PER_CHUNK = 350    # CẢI TIẾN: 200 → 350 (tận dụng MAX_SEQ_LENGTH=2048 tốt hơn)
CHUNK_OVERLAP = 50           # CẢI TIẾN: 30 → 50
INCLUDE_APPENDIX = True
INCLUDE_PREAMBLE = True      # MỚI: thêm chunk cho phần preamble (trước Điều 1)

# ======================== RETRIEVAL ========================
FUSION_METHOD = "rrf"
ALPHA = 0.9
RRF_K = 60
HYBRID_TOP_N = 80            # CẢI TIẾN: 30 → 80 (giảm retrieval miss)
NUM_ANSWERS = 5

# ======================== RERANKING (CẢI TIẾN) ========================
RERANK_ALL_CHUNKS = True     # MỚI: rerank TẤT CẢ chunks, dedupe SAU rerank
MULTI_CHUNK_BONUS = 0.02     # MỚI: bonus cho doc có nhiều chunk match
LAMBDA_WEIGHT = 0.2

# ======================== QUERY EXPANSION (MỚI) ========================
USE_QUERY_EXPANSION = True   # MỚI: PRF-based query expansion cho BM25
PRF_TOP_K = 3                # Số chunk dùng cho pseudo-relevance feedback
PRF_MAX_KEYWORDS = 10        # Số từ khóa tối đa bổ sung

# ======================== BATCH SIZES ========================
ENCODE_BATCH_SIZE = 64
RERANK_BATCH_SIZE = 128
QUESTION_CHUNK_SIZE = 50     # Giảm từ 100 vì HYBRID_TOP_N lớn hơn

print("Thiết bị dùng:", GPU_DEVICES)
print("Index sẽ đọc từ:", resolve_index_read_dir())

Thiết bị dùng: ['cuda:0', 'cuda:1']
Index sẽ đọc từ: /kaggle/input/datasets/thinhng04/result-cache


## Chunking (CẢI TIẾN)
> - Thêm **preamble chunk** (phần trước Điều 1) chứa metadata văn bản
> - Chunk size 350 words, overlap 50 words
> - MAX_SEQ_LENGTH = 2048 để không bị cắt cụt

In [5]:
_PROVINCES = [
    "Thành phố Hồ Chí Minh", "TP. Hồ Chí Minh", "TP Hồ Chí Minh", "Hồ Chí Minh",
    "Hà Nội", "Hải Phòng", "Đà Nẵng", "Cần Thơ",
    "An Giang", "Bà Rịa - Vũng Tàu", "Bà Rịa-Vũng Tàu", "Bạc Liêu", "Bắc Giang", "Bắc Kạn",
    "Bắc Ninh", "Bến Tre", "Bình Dương", "Bình Định", "Bình Phước", "Bình Thuận",
    "Cà Mau", "Cao Bằng", "Đắk Lắk", "Đắk Nông", "Điện Biên", "Đồng Nai", "Đồng Tháp",
    "Gia Lai", "Hà Giang", "Hà Nam", "Hà Tĩnh", "Hải Dương", "Hậu Giang", "Hòa Bình",
    "Hưng Yên", "Khánh Hòa", "Kiên Giang", "Kon Tum", "Lai Châu", "Lâm Đồng", "Lạng Sơn",
    "Lào Cai", "Long An", "Nam Định", "Nghệ An", "Ninh Bình", "Ninh Thuận", "Phú Thọ",
    "Phú Yên", "Quảng Bình", "Quảng Nam", "Quảng Ngãi", "Quảng Ninh", "Quảng Trị",
    "Sóc Trăng", "Sơn La", "Tây Ninh", "Thái Bình", "Thái Nguyên", "Thanh Hóa",
    "Thừa Thiên Huế", "Tiền Giang", "Trà Vinh", "Tuyên Quang", "Vĩnh Long", "Vĩnh Phúc", "Yên Bái",
]
_PROVINCES_SORTED = sorted(set(_PROVINCES), key=len, reverse=True)
PROVINCE_PATTERN = "|".join(re.escape(p) for p in _PROVINCES_SORTED)
ADMIN_PREFIX = r"(?:Tỉnh\s+|Thành\s+phố\s+|TP\.?\s*)?"


def extract_title(passage, search_end_pos=1000):
    preamble = passage[:search_end_pos]
    normalized = re.sub(r"\s+", " ", preamble).strip()

    match = re.search(
        r"(THÔNG TƯ LIÊN TỊCH|THÔNG TƯ|NGHỊ ĐỊNH|NGHỊ QUYẾT|QUYẾT ĐỊNH|CHỈ THỊ|"
        r"PHÁP LỆNH|LUẬT|HƯỚNG DẪN|QUY CHUẨN|QUY ĐỊNH|THÔNG BÁO|KẾ HOẠCH|"
        r"TỜ TRÌNH|BÁO CÁO|CÔNG ĐIỆN|CÔNG VĂN)\s+(.*?)(?=Căn cứ|Theo đề nghị|$)",
        normalized
    )
    if match:
        title = match.group(2).strip()
        if title:
            return f"{match.group(1)}: {title}"

    vv_match = re.search(
        rf"V/v[:\.]?\s*(.*?)(?=(?:{ADMIN_PREFIX}(?:{PROVINCE_PATTERN}))\s*,?\s*ngày\s+\d+\s+tháng\s+\d+\s+năm\s+\d+|Kính gửi|$)",
        normalized
    )
    if vv_match:
        title = vv_match.group(1).strip()
        if title:
            return f"Công văn: {title}"

    return None


def split_into_word_windows(text, title, max_words, overlap):
    words = text.split()
    if len(words) <= max_words:
        if title:
            return [f"{title}\n{text.strip()}"]
        return [text.strip()]

    windows = []
    step = max_words - overlap
    for start in range(0, len(words), step):
        window_text = " ".join(words[start:start + max_words])
        if title:
            windows.append(f"{title}\n{window_text}")
        else:
            windows.append(window_text)
    return windows


def chunk_document(passage, doc_id, max_words=MAX_WORDS_PER_CHUNK, overlap=CHUNK_OVERLAP,
                   include_appendix=INCLUDE_APPENDIX, include_preamble=INCLUDE_PREAMBLE):
    """CẢI TIẾN: Thêm preamble chunk cho phần trước Điều 1."""
    article_positions = [match.start() for match in re.finditer(r"Điều\s+\d+\.", passage)]

    if len(article_positions) == 0:
        title = extract_title(passage, search_end_pos=1000)
        windows = split_into_word_windows(passage, title, max_words, overlap)
        chunks = []
        for window_text in windows:
            chunks.append({"doc_id": doc_id, "text": window_text})
        return chunks

    title = extract_title(passage, search_end_pos=article_positions[0])

    # ===== MỚI: Preamble chunk (phần trước Điều 1) =====
    chunks = []
    if include_preamble:
        preamble_text = passage[:article_positions[0]].strip()
        if preamble_text and len(preamble_text.split()) > 20:
            preamble_windows = split_into_word_windows(preamble_text, title, max_words, overlap)
            for window_text in preamble_windows:
                chunks.append({"doc_id": doc_id, "text": window_text})

    # ===== Phần body (các Điều) =====
    appendix_match = re.search(r"PHỤ LỤC", passage)
    body_end = appendix_match.start() if appendix_match else len(passage)
    body = passage[article_positions[0]:body_end]

    article_positions_in_body = [match.start() for match in re.finditer(r"Điều\s+\d+\.", body)]
    article_positions_in_body.append(len(body))

    for i in range(len(article_positions_in_body) - 1):
        segment = body[article_positions_in_body[i]:article_positions_in_body[i + 1]].strip()
        if len(segment) == 0:
            continue
        if len(segment.split()) > max_words:
            windows = split_into_word_windows(segment, title, max_words, overlap)
        elif title:
            windows = [f"{title}\n{segment}"]
        else:
            windows = [segment]
        for window_text in windows:
            chunks.append({"doc_id": doc_id, "text": window_text})

    if include_appendix and appendix_match:
        appendix_text = passage[appendix_match.start():]
        appendix_positions = [match.start() for match in re.finditer(r"PHỤ LỤC[^\n]{0,40}", appendix_text)]
        appendix_positions.append(len(appendix_text))
        for i in range(len(appendix_positions) - 1):
            segment = appendix_text[appendix_positions[i]:appendix_positions[i + 1]].strip()
            if len(segment) == 0:
                continue
            windows = split_into_word_windows(segment, title, max_words, overlap)
            for window_text in windows:
                chunks.append({"doc_id": doc_id, "text": window_text})

    return chunks

## Corpus Building & Indexing

In [6]:
def load_context_records(context_dir):
    records = []
    files = sorted(glob.glob(os.path.join(context_dir, "context_*.json")))
    for file_path in tqdm(files, desc="Loading context files"):
        with open(file_path, encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            records.extend(data)
        else:
            records.append(data)
    return records


def build_corpus(context_dir):
    records = load_context_records(context_dir)
    corpus_texts = []
    corpus_doc_ids = []
    for record in tqdm(records, desc="Chunking documents"):
        doc_chunks = chunk_document(record["passage"], str(record["id"]))
        for chunk in doc_chunks:
            corpus_texts.append(chunk["text"])
            corpus_doc_ids.append(chunk["doc_id"])
    return corpus_texts, corpus_doc_ids


def tokenize_for_bm25(text):
    segmented_text = ViTokenizer.tokenize(str(text))
    cleaned_text = re.sub(r"[^\w\s]", " ", segmented_text)
    return cleaned_text.lower().split()


def build_bm25(corpus_texts):
    tokenized_corpus = []
    for text in tqdm(corpus_texts, desc="Tokenizing for BM25"):
        tokenized_corpus.append(tokenize_for_bm25(text))
    return BM25Okapi(tokenized_corpus)


def encode_corpus(dense_model, corpus_texts):
    if len(GPU_DEVICES) > 1:
        pool = dense_model.start_multi_process_pool(target_devices=GPU_DEVICES)
        embeddings = dense_model.encode_multi_process(corpus_texts, pool, batch_size=ENCODE_BATCH_SIZE)
        dense_model.stop_multi_process_pool(pool)
        embeddings = torch.tensor(embeddings)
    else:
        embeddings = dense_model.encode(
            corpus_texts, batch_size=ENCODE_BATCH_SIZE,
            convert_to_tensor=True, device=PRIMARY_DEVICE,
            show_progress_bar=True,
        )
    return embeddings


def save_index(corpus_texts, corpus_doc_ids, bm25, embeddings):
    with open(os.path.join(INDEX_DIR, "corpus.pkl"), "wb") as f:
        pickle.dump({"texts": corpus_texts, "doc_ids": corpus_doc_ids}, f)
    with open(os.path.join(INDEX_DIR, "bm25.pkl"), "wb") as f:
        pickle.dump(bm25, f)
    torch.save(embeddings, os.path.join(INDEX_DIR, "dense_embeddings.pt"))
    print(f"Đã lưu index vào {INDEX_DIR}")


def load_index_from(read_dir):
    with open(os.path.join(read_dir, "corpus.pkl"), "rb") as f:
        corpus_data = pickle.load(f)
    with open(os.path.join(read_dir, "bm25.pkl"), "rb") as f:
        bm25 = pickle.load(f)
    embeddings = torch.load(os.path.join(read_dir, "dense_embeddings.pt"), weights_only=False)
    return corpus_data["texts"], corpus_data["doc_ids"], bm25, embeddings


def get_or_build_corpus_and_index(dense_model):
    read_dir = resolve_index_read_dir()
    embeddings_file = os.path.join(read_dir, "dense_embeddings.pt")

    if os.path.exists(embeddings_file):
        print(f"Tìm thấy index có sẵn tại {read_dir} — nạp lại.")
        return load_index_from(read_dir)

    print("Không tìm thấy index có sẵn — build mới.")
    corpus_texts, corpus_doc_ids = build_corpus(CONTEXT_DIR)
    bm25 = build_bm25(corpus_texts)
    embeddings = encode_corpus(dense_model, corpus_texts)
    save_index(corpus_texts, corpus_doc_ids, bm25, embeddings)
    return corpus_texts, corpus_doc_ids, bm25, embeddings

In [7]:
dense_model = SentenceTransformer(DENSE_MODEL_NAME, device=PRIMARY_DEVICE)
dense_model.max_seq_length = MAX_SEQ_LENGTH
if PRIMARY_DEVICE.startswith("cuda"):
    dense_model = dense_model.half()

corpus_texts, corpus_doc_ids, bm25, corpus_embeddings = get_or_build_corpus_and_index(dense_model)
corpus_embeddings = corpus_embeddings.to(PRIMARY_DEVICE)

corpus_doc_id_set = set(corpus_doc_ids)
doc_to_chunk_indices = defaultdict(list)
for idx, doc_id in enumerate(corpus_doc_ids):
    doc_to_chunk_indices[doc_id].append(idx)

print(f"Số chunk trong corpus: {len(corpus_texts)}")
print(f"Số unique doc: {len(doc_to_chunk_indices)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/708 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Tìm thấy index có sẵn tại /kaggle/input/datasets/thinhng04/result-cache — nạp lại.
Số chunk trong corpus: 326156
Số unique doc: 8532


## Scoring Functions
> **CẢI TIẾN #5**: Query Expansion (PRF) cho BM25

In [8]:
def bm25_score(question):
    tokens = tokenize_for_bm25(question)
    return bm25.get_scores(tokens).astype(np.float32)


def bm25_score_batch(questions):
    scores = []
    for question in questions:
        scores.append(bm25_score(question))
    return torch.tensor(np.array(scores), device=PRIMARY_DEVICE)


def dense_score_batch(questions):
    query_embeddings = dense_model.encode(
        questions, batch_size=ENCODE_BATCH_SIZE,
        convert_to_tensor=True, device=PRIMARY_DEVICE,
    )
    return util.cos_sim(query_embeddings, corpus_embeddings)


def expanded_bm25_score_batch(questions, dense_scores=None):
    """MỚI: Pseudo-Relevance Feedback.
    Dùng dense retrieval tìm top-K chunks → trích keywords bổ sung cho BM25 query."""
    if not USE_QUERY_EXPANSION:
        return bm25_score_batch(questions)

    if dense_scores is None:
        dense_scores = dense_score_batch(questions)

    scores = []
    for i, question in enumerate(questions):
        top_k_indices = torch.topk(dense_scores[i], PRF_TOP_K).indices.cpu().numpy()
        expansion_words = []
        for idx in top_k_indices:
            chunk_tokens = list(bm25.doc_freqs[idx].keys())
            expansion_words.extend(chunk_tokens)
        original_tokens = tokenize_for_bm25(question)
        combined_tokens = original_tokens + expansion_words[:PRF_MAX_KEYWORDS]
        scores.append(bm25.get_scores(combined_tokens).astype(np.float32))

    return torch.tensor(np.array(scores), device=PRIMARY_DEVICE)


def normalize_scores(score_matrix):
    max_values = score_matrix.max(dim=1, keepdim=True).values
    return score_matrix / (max_values + 1e-9)


def get_score(entry):
    return entry[1]


def dedupe_by_doc_id(chunk_indices, scores, corpus_doc_ids):
    best_score_by_doc = {}
    for chunk_index, score in zip(chunk_indices, scores):
        doc_id = corpus_doc_ids[chunk_index]
        if doc_id not in best_score_by_doc or score > best_score_by_doc[doc_id]:
            best_score_by_doc[doc_id] = score
    return sorted(best_score_by_doc.items(), key=get_score, reverse=True)


def dedupe_by_doc_id_agg(chunk_indices, scores, corpus_doc_ids, bonus_per_extra=MULTI_CHUNK_BONUS):
    """CẢI TIẾN: Dedupe với multi-chunk aggregation — bonus cho doc có nhiều chunk match."""
    doc_scores = defaultdict(list)
    for chunk_index, score in zip(chunk_indices, scores):
        doc_id = corpus_doc_ids[chunk_index]
        doc_scores[doc_id].append(float(score))

    final = {}
    for doc_id, score_list in doc_scores.items():
        max_score = max(score_list)
        bonus = bonus_per_extra * (len(score_list) - 1)
        final[doc_id] = max_score + bonus

    return sorted(final.items(), key=lambda x: x[1], reverse=True)


def ranks_from_scores(score_matrix):
    order = torch.argsort(score_matrix, dim=1, descending=True)
    ranks = torch.empty_like(order)
    arange = torch.arange(score_matrix.shape[1], device=score_matrix.device).expand_as(order)
    ranks.scatter_(1, order, arange)
    return ranks


def hybrid_candidates(questions, alpha=ALPHA, top_n=HYBRID_TOP_N,
                       fusion_method=FUSION_METHOD, rrf_k=RRF_K):
    """CẢI TIẾN: Dùng expanded BM25 nếu bật query expansion."""
    dense_scores = dense_score_batch(questions)

    if USE_QUERY_EXPANSION:
        bm25_scores = expanded_bm25_score_batch(questions, dense_scores=dense_scores)
    else:
        bm25_scores = bm25_score_batch(questions)

    if fusion_method == "rrf":
        bm25_ranks = ranks_from_scores(bm25_scores)
        dense_ranks = ranks_from_scores(dense_scores)
        final_scores = 1.0 / (rrf_k + bm25_ranks.float() + 1) + 1.0 / (rrf_k + dense_ranks.float() + 1)
    else:
        bm25_normalized = normalize_scores(bm25_scores)
        dense_normalized = normalize_scores(dense_scores)
        final_scores = alpha * dense_normalized + (1 - alpha) * bm25_normalized

    top_values, top_indices = torch.topk(final_scores, top_n, dim=1)
    return top_indices.cpu().numpy(), top_values.cpu().numpy()

## Evaluation & Alpha Tuning

In [9]:
def precision_recall(predicted_doc_ids, gold_doc_ids, max_answers=NUM_ANSWERS):
    predicted_set = set(predicted_doc_ids)
    if len(predicted_set) == 0 or len(predicted_set) > max_answers:
        return 0.0, 0.0
    true_positives = len(predicted_set & gold_doc_ids)
    precision = true_positives / len(predicted_set)
    recall = true_positives / len(gold_doc_ids) if len(gold_doc_ids) > 0 else 0.0
    return precision, recall


with open(TRAIN_FILE, encoding="utf-8") as f:
    train_data = json.load(f)

# ĐÃ SỬA: Chỉ dùng 800 câu để tune cho nhanh thay vì toàn bộ
random.seed(42)
sample_size = min(len(train_data), 800)
sample_qids = random.sample(list(train_data.keys()), sample_size)

sample_questions = []
sample_golds = []
for qid in sample_qids:
    sample_questions.append(train_data[qid]["question"])
    sample_golds.append(set(train_data[qid]["answer"]))

print(f"Dùng {len(sample_questions)} câu hỏi train để đánh giá")

Dùng 800 câu hỏi train để đánh giá


In [10]:
# def precompute_bm25_dense(questions, chunk_size=QUESTION_CHUNK_SIZE):
#     cached = []
#     for start in tqdm(range(0, len(questions), chunk_size), desc="Precompute bm25+dense"):
#         end = min(start + chunk_size, len(questions))
#         batch_questions = questions[start:end]
#         d_scores = dense_score_batch(batch_questions)
#         if USE_QUERY_EXPANSION:
#             b_scores = expanded_bm25_score_batch(batch_questions, dense_scores=d_scores)
#         else:
#             b_scores = bm25_score_batch(batch_questions)
#         cached.append((b_scores, d_scores))
#     return cached


# def search_best_alpha_cached(cached, golds, alpha_candidates, top_n=HYBRID_TOP_N):
#     print(f"{'Alpha':<8}{'Precision':>12}{'Recall':>10}")
#     best_alpha, best_recall, best_precision = None, -1.0, None

#     for alpha in alpha_candidates:
#         precision_sum = recall_sum = question_count = 0.0
#         gold_cursor = 0

#         for bm25_scores, dense_scores in cached:
#             batch_size = bm25_scores.shape[0]
#             batch_golds = golds[gold_cursor:gold_cursor + batch_size]
#             gold_cursor += batch_size

#             final_scores = alpha * normalize_scores(dense_scores) + (1 - alpha) * normalize_scores(bm25_scores)
#             top_values, top_indices = torch.topk(final_scores, top_n, dim=1)
#             top_indices_np, top_values_np = top_indices.cpu().numpy(), top_values.cpu().numpy()

#             for i in range(batch_size):
#                 ranked_docs = dedupe_by_doc_id_agg(top_indices_np[i], top_values_np[i], corpus_doc_ids)
#                 predicted = [d for d, _ in ranked_docs[:NUM_ANSWERS]]
#                 p, r = precision_recall(predicted, batch_golds[i])
#                 precision_sum += p; recall_sum += r; question_count += 1

#         avg_p, avg_r = precision_sum / question_count, recall_sum / question_count
#         print(f"{alpha:<8}{avg_p:>12.4f}{avg_r:>10.4f}")
#         if avg_r > best_recall:
#             best_recall, best_alpha, best_precision = avg_r, alpha, avg_p

#     print(f"\nAlpha tối ưu Recall: {best_alpha} (Recall={best_recall:.4f}, Precision={best_precision:.4f})")
#     return best_alpha


# cached_scores = precompute_bm25_dense(sample_questions)
# ALPHA = search_best_alpha_cached(cached_scores, sample_golds, [0.5, 0.6, 0.7, 0.75, 0.8, 0.82, 0.85, 0.9])

## Reranker Setup

In [11]:
reranker_devices = GPU_DEVICES[:2] if len(GPU_DEVICES) > 1 else [PRIMARY_DEVICE]

rerankers = []
for device in reranker_devices:
    rerankers.append(CrossEncoder(RERANKER_MODEL_NAME, max_length=RERANKER_MAX_LENGTH, device=device))

print("Reranker chạy trên:", reranker_devices)


def rerank_pairs(pairs):
    if len(pairs) == 0:
        return np.array([])
    if len(rerankers) == 1:
        return rerankers[0].predict(pairs, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False)

    midpoint = len(pairs) // 2
    first_half = pairs[:midpoint]
    second_half = pairs[midpoint:]

    with ThreadPoolExecutor(max_workers=2) as executor:
        first_future = executor.submit(rerankers[0].predict, first_half, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False)
        second_future = executor.submit(rerankers[1].predict, second_half, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False)
        first_scores = first_future.result()
        second_scores = second_future.result()

    return np.concatenate([first_scores, second_scores])

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Reranker chạy trên: ['cuda:0', 'cuda:1']


## Lambda Tuning (CẢI TIẾN)
> Rerank TẤT CẢ chunks → dedupe SAU rerank → multi-chunk bonus

In [12]:
def minmax_norm(values):
    values = np.asarray(values, dtype=np.float64)
    lo, hi = values.min(), values.max()
    if hi - lo < 1e-9:
        return np.zeros_like(values)
    return (values - lo) / (hi - lo)


def precompute_hybrid_and_rerank(questions, top_n=HYBRID_TOP_N, chunk_size=QUESTION_CHUNK_SIZE):
    """CẢI TIẾN: Rerank TẤT CẢ chunks, KHÔNG dedupe trước rerank."""
    cached = []

    for start in tqdm(range(0, len(questions), chunk_size), desc="Precompute hybrid+rerank"):
        end = min(start + chunk_size, len(questions))
        batch_questions = questions[start:end]

        top_indices, top_values = hybrid_candidates(batch_questions, top_n=top_n)

        if RERANK_ALL_CHUNKS:
            all_pairs = []
            pair_counts = []
            for i in range(len(batch_questions)):
                count = 0
                for idx in top_indices[i]:
                    all_pairs.append([batch_questions[i], corpus_texts[idx]])
                    count += 1
                pair_counts.append(count)

            rerank_scores = rerank_pairs(all_pairs)

            cursor = 0
            for i in range(len(batch_questions)):
                count = pair_counts[i]
                scores_i = rerank_scores[cursor:cursor + count]
                chunk_indices_i = top_indices[i][:count]
                hybrid_scores_i = top_values[i][:count]
                cursor += count

                doc_rerank = defaultdict(list)
                doc_hybrid = defaultdict(list)
                for idx, r_score, h_score in zip(chunk_indices_i, scores_i, hybrid_scores_i):
                    doc_id = corpus_doc_ids[idx]
                    doc_rerank[doc_id].append(float(r_score))
                    doc_hybrid[doc_id].append(float(h_score))

                candidate_doc_ids = list(doc_rerank.keys())
                rerank_scores_agg = []
                hybrid_scores_agg = []
                for d in candidate_doc_ids:
                    bonus = MULTI_CHUNK_BONUS * (len(doc_rerank[d]) - 1)
                    rerank_scores_agg.append(max(doc_rerank[d]) + bonus)
                    hybrid_scores_agg.append(max(doc_hybrid[d]) + bonus)

                cached.append((
                    candidate_doc_ids,
                    hybrid_scores_agg,
                    np.array(rerank_scores_agg)
                ))
        else:
            all_pairs = []
            batch_meta = []
            for i in range(len(batch_questions)):
                best_chunk_by_doc = {}
                for idx, score in zip(top_indices[i].tolist(), top_values[i].tolist()):
                    doc_id = corpus_doc_ids[idx]
                    if doc_id not in best_chunk_by_doc or score > best_chunk_by_doc[doc_id][1]:
                        best_chunk_by_doc[doc_id] = (idx, score)

                candidate_doc_ids = list(best_chunk_by_doc.keys())
                hybrid_scores_raw = [best_chunk_by_doc[d][1] for d in candidate_doc_ids]
                for d in candidate_doc_ids:
                    all_pairs.append([batch_questions[i], corpus_texts[best_chunk_by_doc[d][0]]])
                batch_meta.append((candidate_doc_ids, hybrid_scores_raw, len(candidate_doc_ids)))

            rerank_scores = rerank_pairs(all_pairs)

            cursor = 0
            for candidate_doc_ids, hybrid_scores_raw, count in batch_meta:
                scores_i = rerank_scores[cursor:cursor + count]
                cursor += count
                cached.append((candidate_doc_ids, hybrid_scores_raw, scores_i))

    return cached


def predictions_from_cache(cached, num_answers=NUM_ANSWERS, lambda_weight=1.0):
    results = []
    for candidate_doc_ids, hybrid_scores_raw, rerank_scores_raw in cached:
        rerank_norm = minmax_norm(rerank_scores_raw)
        hybrid_norm = minmax_norm(hybrid_scores_raw)
        combined = lambda_weight * rerank_norm + (1 - lambda_weight) * hybrid_norm
        order = np.argsort(combined)[::-1]
        results.append([candidate_doc_ids[j] for j in order[:num_answers]])
    return results


def search_best_rerank_blend(questions, golds, lambda_candidates, top_n=HYBRID_TOP_N, chunk_size=QUESTION_CHUNK_SIZE):
    cached = precompute_hybrid_and_rerank(questions, top_n=top_n, chunk_size=chunk_size)

    print(f"{'Lambda':<8}{'Precision':>12}{'Recall':>10}")
    best_lambda, best_recall, best_precision = None, -1.0, None

    for lambda_weight in lambda_candidates:
        predicted_lists = predictions_from_cache(cached, lambda_weight=lambda_weight)

        precision_sum = 0.0
        recall_sum = 0.0
        for i in range(len(questions)):
            precision, recall = precision_recall(predicted_lists[i], golds[i])
            precision_sum += precision
            recall_sum += recall

        average_precision = precision_sum / len(questions)
        average_recall = recall_sum / len(questions)
        print(f"{lambda_weight:<8}{average_precision:>12.4f}{average_recall:>10.4f}")

        if average_recall > best_recall:
            best_recall, best_lambda, best_precision = average_recall, lambda_weight, average_precision

    print(f"\nLambda tối ưu Recall: {best_lambda} (Recall={best_recall:.4f}, Precision={best_precision:.4f})")
    return best_lambda


best_lambda = search_best_rerank_blend(
    sample_questions, sample_golds,
    [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15, 0.1]
)
LAMBDA_WEIGHT = best_lambda

Precompute hybrid+rerank:   0%|          | 0/16 [00:00<?, ?it/s]

Lambda     Precision    Recall
1.0           0.2044    0.9523
0.9           0.2044    0.9523
0.8           0.2036    0.9485
0.7           0.2034    0.9473
0.6           0.2024    0.9435
0.5           0.2019    0.9420
0.4           0.2011    0.9395
0.35          0.2006    0.9376
0.3           0.2004    0.9372
0.25          0.2006    0.9391
0.2           0.1989    0.9311
0.15          0.1981    0.9268
0.1           0.1956    0.9149

Lambda tối ưu Recall: 1.0 (Recall=0.9523, Precision=0.2044)


## Run Pipeline

In [13]:
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)


def load_questions(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_checkpoint(path):
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        print(f"Checkpoint có sẵn: {len(data)} câu đã có kết quả.")
        return data
    return {}


def save_checkpoint(submission, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)


def run_pipeline(questions_data, checkpoint_path, lambda_weight=LAMBDA_WEIGHT):
    submission = load_checkpoint(checkpoint_path)
    remaining_qids = [qid for qid in questions_data if qid not in submission]
    print(f"Còn {len(remaining_qids)}/{len(questions_data)} câu hỏi cần chạy.")

    for start in tqdm(range(0, len(remaining_qids), QUESTION_CHUNK_SIZE), desc="Running pipeline"):
        end = min(start + QUESTION_CHUNK_SIZE, len(remaining_qids))
        batch_qids = remaining_qids[start:end]
        batch_questions = [questions_data[qid]["question"] for qid in batch_qids]

        top_indices, top_values = hybrid_candidates(batch_questions, alpha=ALPHA, top_n=HYBRID_TOP_N)

        if RERANK_ALL_CHUNKS:
            all_pairs = []
            pair_counts = []
            for i in range(len(batch_questions)):
                count = 0
                for idx in top_indices[i]:
                    all_pairs.append([batch_questions[i], corpus_texts[idx]])
                    count += 1
                pair_counts.append(count)

            rerank_scores = rerank_pairs(all_pairs)

            cursor = 0
            for i, qid in enumerate(batch_qids):
                count = pair_counts[i]
                scores_i = rerank_scores[cursor:cursor + count]
                chunk_indices_i = top_indices[i][:count]
                hybrid_scores_i = top_values[i][:count]
                cursor += count

                doc_rerank = defaultdict(list)
                doc_hybrid = defaultdict(list)
                for idx, r_score, h_score in zip(chunk_indices_i, scores_i, hybrid_scores_i):
                    doc_id = corpus_doc_ids[idx]
                    doc_rerank[doc_id].append(float(r_score))
                    doc_hybrid[doc_id].append(float(h_score))

                candidate_doc_ids = list(doc_rerank.keys())
                rerank_agg = []
                hybrid_agg = []
                for d in candidate_doc_ids:
                    bonus = MULTI_CHUNK_BONUS * (len(doc_rerank[d]) - 1)
                    rerank_agg.append(max(doc_rerank[d]) + bonus)
                    hybrid_agg.append(max(doc_hybrid[d]) + bonus)

                rerank_norm = minmax_norm(np.array(rerank_agg))
                hybrid_norm = minmax_norm(np.array(hybrid_agg))
                combined = lambda_weight * rerank_norm + (1 - lambda_weight) * hybrid_norm

                order = np.argsort(combined)[::-1]
                answer_doc_ids = [candidate_doc_ids[j] for j in order[:NUM_ANSWERS]]
                submission[qid] = {"answer": answer_doc_ids}

        else:
            all_pairs = []
            batch_meta = []
            for i in range(len(batch_questions)):
                best_chunk_by_doc = {}
                for idx, score in zip(top_indices[i].tolist(), top_values[i].tolist()):
                    doc_id = corpus_doc_ids[idx]
                    if doc_id not in best_chunk_by_doc or score > best_chunk_by_doc[doc_id][1]:
                        best_chunk_by_doc[doc_id] = (idx, score)

                candidate_doc_ids = list(best_chunk_by_doc.keys())
                hybrid_scores_raw = [best_chunk_by_doc[d][1] for d in candidate_doc_ids]
                for d in candidate_doc_ids:
                    all_pairs.append([batch_questions[i], corpus_texts[best_chunk_by_doc[d][0]]])
                batch_meta.append((candidate_doc_ids, hybrid_scores_raw, len(candidate_doc_ids)))

            rerank_scores = rerank_pairs(all_pairs)

            cursor = 0
            for i, qid in enumerate(batch_qids):
                candidate_doc_ids, hybrid_scores_raw, count = batch_meta[i]
                rerank_scores_raw = rerank_scores[cursor:cursor + count]
                cursor += count

                rerank_norm = minmax_norm(rerank_scores_raw)
                hybrid_norm = minmax_norm(hybrid_scores_raw)
                combined = lambda_weight * rerank_norm + (1 - lambda_weight) * hybrid_norm

                order = np.argsort(combined)[::-1]
                answer_doc_ids = [candidate_doc_ids[j] for j in order[:NUM_ANSWERS]]
                submission[qid] = {"answer": answer_doc_ids}

        save_checkpoint(submission, checkpoint_path)
        torch.cuda.empty_cache()
        gc.collect()
    return submission

In [14]:
INPUT_FILE = PUBLIC_TEST_FILE
questions_data = load_questions(INPUT_FILE)
submission = run_pipeline(questions_data, CHECKPOINT_FILE)

for qid, item in submission.items():
    if len(item["answer"]) == 0 or len(item["answer"]) > NUM_ANSWERS:
        raise ValueError(f"Câu {qid} có số lượng đáp án không hợp lệ: {len(item['answer'])}")

with open(SUBMISSION_FILE, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)
print(f"Đã ghi {len(submission)} câu trả lời vào {SUBMISSION_FILE}")

Còn 1000/1000 câu hỏi cần chạy.


Running pipeline:   0%|          | 0/20 [00:00<?, ?it/s]

Đã ghi 1000 câu trả lời vào /kaggle/working/outputs/submission.json


## Evaluation (Optional)

In [15]:
def eval_retrieval(y_pred, y_true):
    predicted_answers = {}
    for qid, item in y_pred.items():
        predicted_answers[qid] = item["answer"]

    prediction_ids = list(predicted_answers.keys())
    truth_ids = list(y_true.keys())

    if len(prediction_ids) != len(truth_ids):
        raise Exception("Samples in predict not match with reference")

    recall_scores = []
    for qid in truth_ids:
        predicted = predicted_answers.get(qid, set())
        if len(predicted) > 0 and len(predicted) <= 5:
            correct = set(y_true[qid]) & set(predicted)
            recall_scores.append(len(correct) / len(y_true[qid]))
        else:
            recall_scores.append(0)

    precision_scores = []
    for qid in prediction_ids:
        predicted = predicted_answers[qid]
        if len(predicted) > 0 and len(predicted) <= 5:
            correct = set(y_true[qid]) & set(predicted)
            precision_scores.append(len(correct) / len(predicted))
        else:
            precision_scores.append(0)

    recall = sum(recall_scores) / len(recall_scores)
    precision = sum(precision_scores) / len(precision_scores)
    return {"precision": precision, "recall": recall}


def score_against_train(submission, train_file=TRAIN_FILE):
    with open(train_file, encoding="utf-8") as f:
        train_data = json.load(f)

    ground_truth = {}
    for qid, item in train_data.items():
        ground_truth[qid] = item["answer"]

    scores = eval_retrieval(submission, ground_truth)
    print("Final scores:", scores)

    with open(os.path.join(OUTPUT_DIR, "scores.json"), "w") as f:
        json.dump(scores, f)

    return scores


# Chỉ chạy được nếu INPUT_FILE = TRAIN_FILE
# scores = score_against_train(submission, TRAIN_FILE)